# Phase 1a — Data Verification Gate

Proves **OpenAQ** (PM2.5 target) and **Open-Meteo** (weather covariates) have the depth,
freshness, and train/serve parity the project needs — *before* building any pipeline.
Picks the nearest **live reference monitor** per city and prints a PASS/FAIL per station.

In [1]:
import datetime as dt
import os

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ["OPENAQ_API_KEY"]
BASE = "https://api.openaq.org/v3"
HEADERS = {"X-API-Key": API_KEY}
NOW = dt.datetime.now(dt.timezone.utc)

PM25_PARAMETER_ID = 2  # OpenAQ's parameter id for pm25
OM_HOURLY = "temperature_2m,wind_speed_10m,wind_direction_10m,precipitation"

# PASS thresholds for the gate
MAX_LATENCY_H = 48    # a "live" sensor has reported within ~2 days
MAX_MISSING_PCT = 20  # <= 20% gaps over the last 30 days
MIN_HISTORY_YR = 2    # enough to cover multiple wildfire seasons

TARGET_CITIES = {
    "Vancouver": (49.2827, -123.1207),
    "Kelowna": (49.8880, -119.4960),
    "Prince George": (53.9171, -122.7497),
}

In [2]:
# Find the PM2.5 sensor's id inside a station's data.
# we dig out the one that measures pm25. Returns None if there isn't one.
def pm25_sensor_id(location: dict) -> int | None:
    for sensor in location.get("sensors", []):
        if sensor.get("parameter", {}).get("name") == "pm25":
            return sensor["id"]
    return None


# Look up how long a sensor has run and how recently it last reported.
def sensor_lifespan(sensor_id: int) -> dict:
    s = requests.get(f"{BASE}/sensors/{sensor_id}", headers=HEADERS, timeout=30).json()["results"][0]
    # The API gives the sensor's first and last reading times as text; convert
    # them to real datetimes so we can do math on them.
    first = dt.datetime.fromisoformat(s["datetimeFirst"]["utc"].replace("Z", "+00:00"))
    last = dt.datetime.fromisoformat(s["datetimeLast"]["utc"].replace("Z", "+00:00"))
    return {
        "history_years": round((last - first).days / 365, 1),   # total span of data
        "last_utc": last.isoformat(),                            # when it last reported
        "latency_h": round((NOW - last).total_seconds() / 3600, 1),  # hours since that report
    }


# Find the closest station to a point that is still actively reporting PM2.5.
def fetch_nearest_live_pm25_station(lat: float, lon: float, monitors_only: bool = True) -> dict | None:
    # Search for PM2.5 stations within 25 km of the given coordinates.
    params = {"coordinates": f"{lat},{lon}", "radius": 25000, "parameters_id": PM25_PARAMETER_ID, "limit": 100}
    if monitors_only:
        params["monitor"] = "true"  # prefer official government monitors over cheap hobby sensors
    r = requests.get(f"{BASE}/locations", headers=HEADERS, timeout=30, params=params)
    r.raise_for_status()
    # The API won't sort by distance for us, so we sort the results ourselves
    # (nearest first), keeping only ones that have a PM2.5 sensor and a distance.
    candidates = sorted((x for x in r.json()["results"] if pm25_sensor_id(x) and x.get("distance") is not None),
                        key=lambda x: x["distance"])
    # Walk from nearest outward and return the first one that reported recently.
    for loc in candidates:
        if sensor_lifespan(pm25_sensor_id(loc))["latency_h"] <= MAX_LATENCY_H:
            return loc
    return None


# Pull the last 30 days of hourly PM2.5 readings as a table. This is what the
# missing-data check counts, and one copy gets saved as a sample file.
def pm25_recent_hourly(sensor_id: int, days: int = 30) -> pd.DataFrame:
    end = NOW.replace(minute=0, second=0, microsecond=0)  # round down to the hour
    start = end - dt.timedelta(days=days)
    r = requests.get(f"{BASE}/sensors/{sensor_id}/measurements/hourly", headers=HEADERS, timeout=30,
                     params={"datetime_from": start.isoformat(), "datetime_to": end.isoformat(), "limit": 1000})
    r.raise_for_status()
    # Keep just the timestamp and value from each reading.
    return pd.DataFrame([{"time_utc": m["period"]["datetimeTo"]["utc"], "pm25": m["value"]}
                         for m in r.json()["results"]])


# Confirm Open-Meteo gives us usable weather from both of its APIs: the
# historical archive (used for training) and the live forecast (used in
# production).
def openmeteo_check(lat: float, lon: float) -> dict:
    # Grab one week of past weather from the historical archive.
    hist = requests.get("https://historical-forecast-api.open-meteo.com/v1/forecast", timeout=30,
                        params={"latitude": lat, "longitude": lon, "start_date": "2025-06-01",
                                "end_date": "2025-06-07", "hourly": OM_HOURLY, "timezone": "UTC"}).json()
    # Grab the next 3 days of forecast weather from the live API.
    live = requests.get("https://api.open-meteo.com/v1/forecast", timeout=30,
                        params={"latitude": lat, "longitude": lon, "forecast_days": 3,
                                "hourly": OM_HOURLY, "timezone": "UTC"}).json()
    return {
        "hist_hours": len(hist["hourly"]["time"]),          # how many past hours came back
        "live_future_hours": len(live["hourly"]["time"]),   # how many future hours came back
        "vars_match": set(hist["hourly_units"]) == set(live["hourly_units"]),  # same variables in both?
        "live_df": pd.DataFrame(live["hourly"]),            # the live forecast, saved as a sample
    }

In [3]:
# Pick the nearest LIVE reference monitor per city (fall back to any live sensor if no monitor nearby).
chosen = {}
for city, (lat, lon) in TARGET_CITIES.items():
    chosen[city] = (fetch_nearest_live_pm25_station(lat, lon)
                    or fetch_nearest_live_pm25_station(lat, lon, monitors_only=False))

# Verify both sources per station and decide PASS/FAIL.
saved_weather = False
for city, loc in chosen.items():
    if loc is None:
        print(f"{city}: no live PM2.5 station within 25 km -> FAIL\n")
        continue

    sid = pm25_sensor_id(loc)
    lat, lon = loc["coordinates"]["latitude"], loc["coordinates"]["longitude"]
    life = sensor_lifespan(sid)
    df = pm25_recent_hourly(sid)
    missing_pct = round(100 * (1 - len(df) / (30 * 24)), 1)
    om = openmeteo_check(lat, lon)

    ok = (life["latency_h"] <= MAX_LATENCY_H and missing_pct <= MAX_MISSING_PCT
          and life["history_years"] >= MIN_HISTORY_YR and om["vars_match"] and om["live_future_hours"] >= 24)

    print(f"{city}: {loc['name']} -> {'PASS' if ok else 'FAIL'}")
    print(f"  location_id={loc['id']}  sensor_id={sid}  dist={loc['distance']/1000:.1f}km  monitor={loc.get('isMonitor')}")
    print(f"  history={life['history_years']}yr  last={life['last_utc']}  latency={life['latency_h']}h")
    print(f"  PM2.5 missing={missing_pct}% ({len(df)}/720 hrs)")
    print(f"  Open-Meteo hist={om['hist_hours']}h  live_future={om['live_future_hours']}h  vars_match={om['vars_match']}\n")

    if ok:
        slug = city.lower().replace(" ", "_")
        df.to_csv(f"sample_pm25_{slug}.csv", index=False)
        if not saved_weather:
            om["live_df"].to_csv("sample_weather_forecast.csv", index=False)
            saved_weather = True

Vancouver: Vancouver-Clark Driv -> PASS
  location_id=2836782  sensor_id=9146190  dist=4.0km  monitor=True
  history=2.1yr  last=2026-06-29T22:00:00+00:00  latency=1.1h
  PM2.5 missing=1.2% (711/720 hrs)
  Open-Meteo hist=168h  live_future=72h  vars_match=True

Kelowna: Kelowna KLO Road -> FAIL
  location_id=230097  sensor_id=1325038  dist=3.5km  monitor=True
  history=4.9yr  last=2026-06-29T19:00:00+00:00  latency=4.1h
  PM2.5 missing=30.3% (502/720 hrs)
  Open-Meteo hist=168h  live_future=72h  vars_match=True

Prince George: PRG Plaza 400 -> PASS
  location_id=2272  sensor_id=4098  dist=0.6km  monitor=True
  history=10.3yr  last=2026-06-29T20:00:00+00:00  latency=3.1h
  PM2.5 missing=1.7% (708/720 hrs)
  Open-Meteo hist=168h  live_future=72h  vars_match=True

